# MAMUT: Auditable Spaceship Titanic Baseline

This notebook demonstrates an evidence-first tabular classification workflow using the released `mamut` package. It creates a submission file, but it does not submit automatically or claim leaderboard-leading performance.

The project has recorded a best public score of **0.80617** from a post-leaderboard CatBoost development campaign. Because that campaign followed observation of an earlier public score, it is useful engineering evidence rather than an independent final-performance estimate.

## Environment

Kaggle runs Python 3.12, which matches MAMUT's supported runtime. Internet is enabled for this kernel so it can install the tagged PyPI release. MAMUT pins its validated ML stack; unrelated preinstalled Kaggle packages are not used by this notebook.

In [ ]:
import subprocess
import sys
import warnings
from importlib.metadata import PackageNotFoundError, version

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="Found unknown categories in columns.*")

requested_version = "0.3.0"
try:
    installed_version = version("mamut")
except PackageNotFoundError:
    installed_version = None
if installed_version != requested_version:
    install = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-q", f"mamut=={requested_version}"],
        capture_output=True,
        text=True,
    )
    if install.returncode != 0:
        raise RuntimeError(f"Unable to install MAMUT from PyPI:\n{install.stderr}")

from mamut import Mamut, __version__

print(f"MAMUT version: {__version__}")

## Load the competition data

The competition dataset is attached through Kaggle's official data source. Passenger groups are retained only for group-disjoint evaluation boundaries.

In [ ]:
from pathlib import Path

import pandas as pd

required_files = ("train.csv", "test.csv")
data_candidates = [
    Path("/kaggle/input/spaceship-titanic"),
    Path("/kaggle/input/competitions/spaceship-titanic"),
]
data_candidates.extend(
    [
        root / ".cache" / "mamut" / "kaggle" / "spaceship-titanic"
        for root in (Path.cwd(), *Path.cwd().parents)
    ]
)
data_dir = next(
    (candidate for candidate in data_candidates if all((candidate / name).exists() for name in required_files)),
    None,
)
if data_dir is None:
    input_root = Path("/kaggle/input")
    available_csv = sorted(str(path) for path in input_root.rglob("*.csv")) if input_root.exists() else []
    raise FileNotFoundError(
        "Could not find the attached Spaceship Titanic train/test files. "
        f"Available Kaggle CSV inputs: {available_csv[:10]}"
    )
print(f"Using competition data from: {data_dir}")
train = pd.read_csv(data_dir / "train.csv")
test = pd.read_csv(data_dir / "test.csv")
passenger_groups = train["PassengerId"].str.split("_").str[0]

train.shape, test.shape, train["Transported"].value_counts(normalize=True).round(3)

## Target-free feature construction

All derived variables use only feature values available at inference time. No target aggregate, leaderboard feedback, or submission output is used for feature generation.

In [ ]:
SPEND_COLUMNS = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]


def make_features(frame: pd.DataFrame) -> pd.DataFrame:
    features = frame.copy()
    features["PassengerGroup"] = features["PassengerId"].str.split("_").str[0]
    features["PassengerNumber"] = features["PassengerId"].str.split("_").str[1].astype(float)
    features["GroupSize"] = features.groupby("PassengerGroup")["PassengerId"].transform("size")
    cabin_parts = features["Cabin"].str.split("/", expand=True)
    features["CabinDeck"] = cabin_parts[0]
    features["CabinNumber"] = pd.to_numeric(cabin_parts[1], errors="coerce")
    features["CabinSide"] = cabin_parts[2]
    features["Surname"] = features["Name"].str.rsplit(" ", n=1).str[-1]
    features["TotalSpend"] = features[SPEND_COLUMNS].fillna(0).sum(axis=1)
    features["NoSpend"] = features["TotalSpend"].eq(0)
    return features.drop(columns=["PassengerId", "Cabin", "Name"])


X = make_features(train.drop(columns="Transported"))
y = train["Transported"].astype(bool)
X_submission = make_features(test)
X.shape, X_submission.shape

## Fit with held-out, group-aware evidence

A single CatBoost family is used here to keep the notebook runtime bounded and interpretable. MAMUT performs the tuning only on modelling data; the reserved holdout is used for final diagnostics rather than candidate selection. Passenger-group separation matches this competition setting, but surnames may recur across splits; this is not an unseen-household generalization claim. Evidence baselines can encounter unseen categorical levels in validation folds; MAMUT's documented one-hot path handles them as unknown values.

In [ ]:
mamut = Mamut(
    include_models=["CatBoostClassifier"],
    score_metric="accuracy",
    optimization_method="random_search",
    n_iterations=3,
    validation_size=0.2,
    holdout_size=0.2,
    refit_final_model=True,
    evidence_cv_splits=3,
    evidence_cv_repeats=1,
    n_jobs=-1,
    random_state=42,
)
mamut.fit(X, y, groups=passenger_groups)
mamut.holdout_summary_.round(4)

In [ ]:
evidence = mamut.generate_evidence(dataset="holdout", include_candidate_comparison=False)
evidence["validation_integrity"], evidence["baseline_comparison"].round(4)

## Create a submission file

The trained public prediction pipeline is applied to competition test rows. Writing the CSV is deliberate; uploading a competition submission remains a separate user action.

In [ ]:
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Transported": mamut.predict(X_submission).astype(bool),
})
submission.to_csv("submission.csv", index=False)
submission.head()

## Interpretation

A credible result is more than one public leaderboard number. Inspect the holdout score, its relationship to simple baselines, group-disjoint validation assumptions, and any later public score separately. For the stronger campaign-controlled evaluation protocol used by the MAMUT repository, see the package documentation and `scripts/benchmark_kaggle.py`.